In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [2]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession baru
spark = SparkSession.builder \
    .appName("Tugas4_BigData") \
    .getOrCreate()

26/10/07 18:22:07 WARN Utils: Your hostname, adnan resolves to a loopback address: 127.0.1.1; using 192.168.1.14 instead (on interface wlp2s0)
26/10/07 18:22:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/07 18:22:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Path wajib menggunakan URL penuh HDFS sesuai instruksi
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"

# 1. Baca dataset langsung dari HDFS
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# 2. Tampilkan Schema
df.printSchema()

# 3. Hitung jumlah baris
print(f"Jumlah baris dataset: {df.count()}")

# 4. Tampilkan 10 baris pertama
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris dataset: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|

In [4]:
from pyspark.sql.functions import col, mean, round as _round

# 1. Tampilkan jumlah data kosong pada kolom rating
null_count = df.filter(col("rating").isNull()).count()
print(f"Jumlah data kosong pada kolom rating: {null_count}")

# 2. Hitung nilai rata-rata rating
mean_rating = df.select(mean(col("rating"))).collect()[0][0]

# 3. Isi nilai null dengan rata-rata yang dibulatkan (2 desimal)
df_clean = df.na.fill({"rating": round(mean_rating, 2)})

# Verifikasi sisa data kosong
print(f"Sisa data kosong pada rating: {df_clean.filter(col('rating').isNull()).count()}")

Jumlah data kosong pada kolom rating: 204
Sisa data kosong pada rating: 0


In [5]:
from pyspark.sql.functions import col, when

# Tambah total_pendapatan dan tier_transaksi
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", 
                                     when(col("total_pendapatan") > 500000, "Besar")
                                     .otherwise("Kecil"))

# Tampilkan sampel data hasil transformasi
df_transformed.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [6]:
from pyspark.sql.functions import sum as _sum, count, avg, round as _round

# 1. Kategori dengan total_pendapatan tertinggi
print("=== Kategori dengan Total Pendapatan Tertinggi ===")
df_transformed.groupBy("kategori") \
    .agg(_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("=== Kota dengan Transaksi Tier 'Besar' Terbanyak ===")
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("tier_transaksi").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
print("=== Rata-Rata Rating per Metode Pembayaran ===")
df_transformed.groupBy("metode_pembayaran") \
    .agg(_round(avg("rating"), 2).alias("rata_rata_rating")) \
    .show()

=== Kategori dengan Total Pendapatan Tertinggi ===
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

=== Kota dengan Transaksi Tier 'Besar' Terbanyak ===
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

=== Rata-Rata Rating per Metode Pembayaran ===
+-----------------+----------------+
|metode_pembayaran|rata_rata_rating|
+-----------------+----------------+
|              COD|            4.17|
|    Transfer Bank|            4.16|
|     Kartu Kredit|            4.12|
|         E-Wallet|            4.14|
+-----------------+----------------+



In [7]:
output_hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_transformed.csv"

# 1. Simpan DataFrame hasil transformasi ke HDFS
df_transformed.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_hdfs_path)

print("Berhasil menyimpan file ke HDFS!")

# 2. Verifikasi dengan membaca kembali data dari HDFS
df_verify = spark.read.csv(output_hdfs_path, header=True, inferSchema=True)

df_verify.printSchema()
df_verify.show(5)

Berhasil menyimpan file ke HDFS!
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- total_pendapatan: integer (nullable = true)
 |-- tier_transaksi: string (nullable = true)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|     